# Plotly figures and a first Dash app
Companion to lecture 2. Run the cells in order. The data is `sales.parquet`, written at the end of lecture 1.

In [1]:
try:
    import dash, dash_bootstrap_components
except ImportError:
    %pip install -q dash dash-bootstrap-components

## Load the sales table

In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

URL = 'https://raw.githubusercontent.com/iitm-da/da2402/master/data%20visualisation/data/sales.parquet'
df = pd.read_parquet(URL)
df.shape

(500, 9)

In [3]:
df.dtypes

Date        datetime64[ns]
Product           category
Region            category
Category          category
Sales                int64
Quantity             int64
Profit               int64
Month               object
Year                 int32
dtype: object

In [4]:
df.head()

,Date,Product,Region,Category,Sales,Quantity,Profit,Month,Year
0,2023-09-28,Mouse,North,Accessories,3872,21,1371,2023-09,2023
1,2023-08-03,Monitor,East,Electronics,4526,11,835,2023-08,2023
2,2024-10-25,Headphones,West,Accessories,230,22,204,2024-10,2024
3,2023-12-10,Headphones,South,Accessories,2533,21,678,2023-12,2023
4,2023-11-10,Laptop,West,Electronics,4217,44,550,2023-11,2023


## A figure is a dict
A Plotly figure holds `data` (a list of traces) and `layout` (everything else).

In [5]:
by_product = (df.groupby('Product', observed=True)['Sales'].sum()
                .sort_values(ascending=False).reset_index())
fig = px.bar(by_product, x='Product', y='Sales')
fig.to_dict().keys()

dict_keys(['data', 'layout'])

In [6]:
len(fig.data), fig.data[0].type

(1, 'bar')

In [7]:
list(fig.data[0].x), fig.data[0].y.tolist()

(['Headphones', 'Laptop', 'Mouse', 'Monitor', 'Keyboard'],
 [296452, 272296, 262878, 255635, 228904])

In [8]:
fig.show()

## The same bar with graph_objects
`px` builds the traces for you. `go` takes them one at a time.

In [9]:
fig_go = go.Figure(go.Bar(x=by_product['Product'], y=by_product['Sales']))
list(fig_go.data[0].x) == list(fig.data[0].x)

True

## Changing the layout and the traces

In [10]:
fig.update_layout(title='Sales by product', yaxis_title='Sales')
fig.update_traces(marker_color='#125F9E')
fig.layout.title.text

'Sales by product'

## Line and scatter
`px.line` draws a scatter trace with lines.

In [11]:
monthly = df.groupby('Month')['Sales'].sum().reset_index()
fig_line = px.line(monthly, x='Month', y='Sales', markers=True)
fig_line.data[0].type, fig_line.data[0].mode

('scatter', 'lines+markers')

In [12]:
fig_sc = px.scatter(df, x='Sales', y='Profit', color='Category')
len(fig_sc.data), [t.name for t in fig_sc.data], [len(t.x) for t in fig_sc.data]

(2, ['Accessories', 'Electronics'], [303, 197])

In [13]:
float(round(df['Sales'].corr(df['Profit']), 3))

0.797

In [14]:
fig_sc.show()

## Grammar of graphics: one argument at a time
Colour maps a column to traces, `log_y` changes the scale, `facet_col` splits into panels.

In [15]:
f_color = px.bar(df, x='Region', y='Sales', color='Category')
f_log = px.bar(by_product, x='Product', y='Sales', log_y=True)
f_facet = px.scatter(df, x='Sales', y='Profit', facet_col='Region')
len(f_color.data), f_log.layout.yaxis.type, len(f_facet.data)

(2, 'log', 4)

In [16]:
f_facet.show()

## A first Dash app
The layout is a tree of components. `dcc.Graph` holds a figure.

In [17]:
from dash import Dash, html, dcc

app = Dash(__name__)
app.layout = html.Div([
    html.H1('Sales by product'),
    dcc.Graph(figure=fig),
])
app.run(jupyter_mode='inline', port=8050)

## The 12-column grid
Each `dbc.Row` has 12 columns. A `dbc.Col` takes a width from 1 to 12.

In [18]:
import dash_bootstrap_components as dbc

box = lambda text: html.Div(text, style={'border': '1px solid #888', 'padding': '8px'})
grid = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])
grid.layout = dbc.Container([
    dbc.Row([dbc.Col(box('6'), width=6), dbc.Col(box('6'), width=6)]),
    dbc.Row([dbc.Col(box('4'), width=4)] * 3),
    dbc.Row([dbc.Col(box('8'), width=8), dbc.Col(box('4'), width=4)]),
    dbc.Row([dbc.Col(box('8'), width=8), dbc.Col(box('6 wraps'), width=6)]),
])
grid.run(jupyter_mode='inline', port=8051)

## The static dashboard
Header, four KPI cards, two charts as 8 and 4. Lecture 3 makes it interactive.

In [19]:
by_region = df.groupby('Region', observed=True)['Sales'].sum().reset_index()
fig_month = px.line(monthly, x='Month', y='Sales', markers=True, title='Sales by month')
fig_region = px.bar(by_region, x='Region', y='Sales', title='Sales by region')

def kpi(label, value):
    return dbc.Card(dbc.CardBody([html.P(label), html.H3(value)]))

dash_app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])
dash_app.layout = dbc.Container([
    dbc.Row(dbc.Col(html.H1('Sales 2023-2024'), width=12)),
    dbc.Row([
        dbc.Col(kpi('Total sales', f"{df['Sales'].sum():,}"), width=3),
        dbc.Col(kpi('Orders', f"{len(df):,}"), width=3),
        dbc.Col(kpi('Units', f"{df['Quantity'].sum():,}"), width=3),
        dbc.Col(kpi('Average order', f"{df['Sales'].mean():,.0f}"), width=3),
    ]),
    dbc.Row([
        dbc.Col(dcc.Graph(figure=fig_month), width=8),
        dbc.Col(dcc.Graph(figure=fig_region), width=4),
    ]),
])
dash_app.run(jupyter_mode='inline', port=8052)

## Try
1. Change the region chart to `Profit` by region.
2. Add a fifth KPI card for total profit and make the five cards fit one row.
3. Colour the line chart by `Category`. How many traces does it have?